# K-SPADA Binning Data Preparation

[Project overview](../README.md) · [01 Data preparation](./01_data_preparation.ipynb) · [02 Model training](./02_model_training.ipynb) · [03 Prediction and visualization](./03_prediction_and_visualization.ipynb) · [04 Quantile binning](./04_binning_data_preparation.ipynb)

This tutorial preserves zero expression and maps nonzero values to cell-wise quantile bins for a generic AnnData object.

Run this notebook from the `tutorials/` directory. This independent preprocessing example reads `../../../../TOSICA-main/data/liver_cancer/all_hvg_3000.h5ad` and writes `../outputs/binning/binned_data.h5ad`. Adapt `DATA_DIR` to your own data location.

Executed on 2026-11-10 with **Python (KSPADA_env @ d05 GPU 3)**. The kernel ran on `d05` with GPU 3 selected. This preprocessing step runs on CPU.


In [1]:
from pathlib import Path
import anndata as ad
import numpy as np
import KSPADA

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = Path("../../../../TOSICA-main/data/liver_cancer").resolve()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "binning"
SOURCE_H5AD = DATA_DIR / "all_hvg_3000.h5ad"
OUTPUT_H5AD = OUTPUT_DIR / "binned_data.h5ad"
BIN_COUNT = 8
RANDOM_SEED = 1

## Step 1 — Check zero-preserving quantile binning

**Execution result:** all assertions passed. With 8 bins and random seed 1, the two example rows became:

```text
[0, 1, 2, 4, 5, 7]
[0, 0, 2, 1, 5, 7]
```

Zero expression remained zero; all binned values lie in 0–7.


In [2]:
example_expression = np.array(
    [
        [0.0, 0.2, 0.5, 1.0, 2.0, 4.0],
        [0.0, 0.0, 3.0, 3.0, 6.0, 9.0],
    ],
    dtype=np.float32,
)
example_binned = KSPADA.bin_expression_matrix(
    example_expression,
    bin_count=BIN_COUNT,
    random_seed=RANDOM_SEED,
)
assert np.all(example_binned[example_expression == 0] == 0)
assert example_binned.min() == 0
assert example_binned.max() <= BIN_COUNT - 1

## Step 2 — Bin the AnnData expression matrix

**Execution result:** `../outputs/binning/binned_data.h5ad` was saved with **84,823 cells × 3,000 genes** and an `int64` expression matrix. A full matrix comparison confirmed that all **233,617,016** original zero entries remained zero and that the complete zero pattern was preserved. All values lie in 0–7.


In [3]:
source_adata = ad.read_h5ad(SOURCE_H5AD)
binned_adata = KSPADA.bin_adata(
    source_adata,
    bin_count=BIN_COUNT,
    random_seed=RANDOM_SEED,
    copy=True,
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
binned_adata.write_h5ad(OUTPUT_H5AD)

/home/lixiangyu/anaconda3/envs/KSPADA_env/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


## Notes

- Bin 0 represents zero detected expression.
- Bins 1 through 7 represent the relative rank of nonzero expression within each cell.